# 03 — Faster R-CNN Training
**Drone Detection | ITMO Diploma Thesis**

Faster R-CNN — **эталонная модель точности**.
- Медленнее YOLO (~15 FPS vs ~100 FPS)
- Выше mAP на мелких объектах
- Используем как baseline для сравнения

Архитектура: ResNet-50 + FPN backbone, RPN, RoI Pooling

**I/O:** чтение тысяч картинок с Google Drive во время эпох сильно замедляет обучение. Ячейка с путями копирует **`prepared/dataset_coco`** в **`/content/drone_coco_local`** (локальный диск сессии Colab). Подготовка из zip не нужна — достаточно уже собранной папки COCO на Drive.

**Метрики / артефакты:** `rcnn_metrics.json` — mAP, **per-class AP** (pycocotools), **recall = COCO AR@100** (не то же самое, что Precision/Recall в Ultralytics). Чекпоинты: **`faster_rcnn_best.pth`**, **`faster_rcnn_last.pth`** (каждая эпоха).

In [ ]:
# ── CELL 1: Install & check GPU ────────────────────────────────────────────────
!uv pip install torch torchvision pycocotools albumentations tqdm matplotlib
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU! Switch to GPU runtime.'
print(f'GPU: {torch.cuda.get_device_name(0)}')
import torchvision; print(f'torchvision: {torchvision.__version__}')

In [ ]:
# ── CELL 2: Mount Drive ───────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── CELL 3: Imports & paths — COCO: Drive → локальная копия ───────────────────
import os
import time
import json
import math
import shutil
from pathlib import Path

import torch
import torch.optim as optim
import torchvision
from torchvision import transforms
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2

DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks')
DRIVE_COCO = DRIVE_ROOT / 'prepared' / 'dataset_coco'
WEIGHTS_DIR = DRIVE_ROOT / 'weights'
RESULTS_DIR = DRIVE_ROOT / 'results'

LOCAL_COCO = Path('/content/drone_coco_local')
FORCE_RECOPY = False  # True — удалить локальную копию и скопировать с Drive заново

WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

_ann_train = DRIVE_COCO / 'annotations' / 'instances_train.json'
assert _ann_train.exists(), (
    f'На Drive нет COCO: {_ann_train}. Скопируйте prepared/dataset_coco с ПК (prepare_merged_dataset.py).'
)

if LOCAL_COCO.exists() and FORCE_RECOPY:
    shutil.rmtree(LOCAL_COCO)

if not LOCAL_COCO.exists():
    print('Копирование dataset_coco: Drive → /content/drone_coco_local …')
    t0 = time.perf_counter()
    shutil.copytree(DRIVE_COCO, LOCAL_COCO)
    print(f'Готово за {time.perf_counter() - t0:.1f} с')
else:
    print('Локальная копия COCO уже есть:', LOCAL_COCO)

COCO_DIR = LOCAL_COCO
print(f'COCO_DIR (fast I/O) = {COCO_DIR}')

DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
CLASS_NAMES = ['DRONE', 'AIRPLANE', 'HELICOPTER', 'BIRD']
NUM_CLASSES = len(CLASS_NAMES) + 1  # +1 for background (index 0)

print(f'Device: {DEVICE}')
print(f'Num classes: {NUM_CLASSES} (including background)')
if torch.cuda.is_available():
    _vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f'GPU VRAM ~{_vram:.1f} GB — при OOM уменьшите batch в DataLoader (CELL 4)')

## 1. Dataset class для COCO format

In [ ]:
# ── CELL 4: COCO Dataset class ────────────────────────────────────────────────


def _parse_coco_bbox(ann: dict) -> tuple[float, float, float, float] | None:
    """COCO bbox [x, y, w, h] → четыре float; None если битые данные."""
    bb = ann.get("bbox")
    if not bb or len(bb) < 4:
        return None
    try:
        x, y, w, h = map(float, bb[:4])
    except (TypeError, ValueError):
        return None
    if w <= 0 or h <= 0:
        return None
    if not all(math.isfinite(v) for v in (x, y, w, h)):
        return None
    return x, y, w, h


def _clip_xyxy_to_image(
    boxes: list[list[float]], labels: list[int], height: int, width: int
) -> tuple[list[list[float]], list[int]]:
    """Pascal VOC xyxy (px): обрезка по кадру, чтобы Albumentations не падал на x_max > 1 после нормализации."""
    out_b: list[list[float]] = []
    out_l: list[int] = []
    w, h = float(width), float(height)
    for box, lab in zip(boxes, labels):
        x1, y1, x2, y2 = (float(v) for v in box)
        x1 = max(0.0, min(x1, w))
        x2 = max(0.0, min(x2, w))
        y1 = max(0.0, min(y1, h))
        y2 = max(0.0, min(y2, h))
        if x2 <= x1 or y2 <= y1:
            continue
        out_b.append([x1, y1, x2, y2])
        out_l.append(lab)
    return out_b, out_l


class DroneCocoDataset(Dataset):
    """COCO: изображения и боксы для torchvision Faster R-CNN."""

    def __init__(self, img_dir: Path, ann_file: Path, transforms=None) -> None:
        self.img_dir = img_dir
        self.transforms = transforms
        with open(ann_file, encoding="utf-8") as f:
            coco = json.load(f)

        # int id: в JSON id / image_id иногда str
        self.images = {int(im["id"]): im for im in coco["images"]}
        self.img_ids = list(self.images.keys())

        self.anns: dict[int, list] = {iid: [] for iid in self.img_ids}
        for ann in coco.get("annotations", []):
            try:
                iid = int(ann["image_id"])
            except (KeyError, TypeError, ValueError):
                continue
            if iid not in self.anns:
                continue
            self.anns[iid].append(ann)

    def __len__(self) -> int:
        return len(self.img_ids)

    def __getitem__(self, idx: int) -> tuple:
        img_id = self.img_ids[idx]
        img_info = self.images[img_id]
        img_path = self.img_dir / img_info["file_name"]
        img = Image.open(img_path).convert("RGB")

        boxes: list[list[float]] = []
        labels: list[int] = []
        for ann in self.anns[img_id]:
            parsed = _parse_coco_bbox(ann)
            if parsed is None:
                continue
            x, y, w, h = parsed
            try:
                cid = int(ann["category_id"])
            except (TypeError, ValueError, KeyError):
                continue
            boxes.append([x, y, x + w, y + h])
            labels.append(cid + 1)  # 0 = background в модели

        img_np = np.array(img)
        ih, iw = img_np.shape[0], img_np.shape[1]
        boxes, labels = _clip_xyxy_to_image(boxes, labels, ih, iw)
        if self.transforms and boxes:
            augmented = self.transforms(image=img_np, bboxes=boxes, labels=labels)
            img_np = augmented["image"]
            boxes = augmented["bboxes"]
            labels = augmented["labels"]
            ih2, iw2 = img_np.shape[0], img_np.shape[1]
            boxes, labels = _clip_xyxy_to_image(boxes, labels, ih2, iw2)

        if not boxes:
            boxes_t = torch.zeros((0, 4), dtype=torch.float32)
            labels_t = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes_t = torch.tensor(boxes, dtype=torch.float32)
            labels_t = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes_t,
            "labels": labels_t,
            "image_id": torch.tensor([img_id]),
            "area": (boxes_t[:, 3] - boxes_t[:, 1]) * (boxes_t[:, 2] - boxes_t[:, 0])
            if len(boxes_t)
            else torch.zeros(0),
            "iscrowd": torch.zeros(len(labels_t), dtype=torch.int64),
        }

        img_tensor = transforms.ToTensor()(Image.fromarray(img_np))
        return img_tensor, target


def collate_fn(batch):
    return tuple(zip(*batch))


train_augment = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, p=0.3),
        A.GaussNoise(std_range=(0.14, 0.34), p=0.2),
        A.MotionBlur(blur_limit=3, p=0.1),
    ],
    bbox_params=A.BboxParams(format="pascal_voc", label_fields=["labels"]),
)

ann_dir = COCO_DIR / "annotations"
train_ds = DroneCocoDataset(
    img_dir=COCO_DIR / "images" / "train",
    ann_file=ann_dir / "instances_train.json",
    transforms=train_augment,
)
val_ds = DroneCocoDataset(
    img_dir=COCO_DIR / "images" / "val",
    ann_file=ann_dir / "instances_val.json",
)

train_loader = DataLoader(
    train_ds,
    batch_size=4,
    shuffle=True,
    num_workers=8,
    collate_fn=collate_fn,
    pin_memory=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=4,
    shuffle=False,
    num_workers=8,
    collate_fn=collate_fn,
    pin_memory=True,
)

print(f"Train: {len(train_ds)} images | Val: {len(val_ds)} images")


## 2. Инициализация модели

In [ ]:
# ── CELL 5: Build Faster R-CNN model ─────────────────────────────────────────
def build_faster_rcnn(num_classes: int) -> torch.nn.Module:
    """Build Faster R-CNN with ResNet-50 FPN, pretrained on COCO.

    Args:
        num_classes: Number of classes including background.

    Returns:
        Faster R-CNN model ready for fine-tuning.
    """
    model = fasterrcnn_resnet50_fpn(
        weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT
    )
    # Replace head for our number of classes
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


model = build_faster_rcnn(NUM_CLASSES)
model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total_params:,}')
print(f'Trainable params: {trainable:,}')

In [ ]:
# ── CELL 6: Optimizer & scheduler ────────────────────────────────────────────
EPOCHS_FREEZE = 5   # Freeze backbone for first N epochs
EPOCHS_TOTAL  = 20

# Phase 1: freeze backbone, train only the head
for name, param in model.named_parameters():
    if 'backbone' in name:
        param.requires_grad = False

params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

print(f'Phase 1: training {len(params)} param groups (backbone frozen)')

## 3. Цикл обучения

После каждой эпохи сохраняется **`faster_rcnn_last.pth`** (последнее состояние + оптимизатор) — можно копировать на Drive и продолжать обучение. Лучшая по `val_loss` модель — **`faster_rcnn_best.pth`**.

In [ ]:
# ── CELL 7: Training loop ─────────────────────────────────────────────────────
def train_one_epoch(model: torch.nn.Module, optimizer: torch.optim.Optimizer,
                    loader: DataLoader, device: torch.device,
                    epoch: int) -> float:
    """Run one training epoch.

    Args:
        model: The detection model.
        optimizer: Optimizer instance.
        loader: Training DataLoader.
        device: Compute device.
        epoch: Current epoch number (for logging).

    Returns:
        Average loss for this epoch.
    """
    model.train()
    total_loss = 0.0
    pbar = tqdm(loader, desc=f'Epoch {epoch:02d} [train]')

    for images, targets in pbar:
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += losses.item()
        pbar.set_postfix({'loss': f'{losses.item():.4f}'})

    return total_loss / len(loader)


@torch.no_grad()
def evaluate_loss(model: torch.nn.Module, loader: DataLoader,
                  device: torch.device) -> float:
    """Compute validation loss (model in train mode for loss computation).

    Args:
        model: The detection model.
        loader: Validation DataLoader.
        device: Compute device.

    Returns:
        Average validation loss.
    """
    model.train()  # Faster R-CNN needs train mode for loss
    total_loss = 0.0
    for images, targets in tqdm(loader, desc='Val loss', leave=False):
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        with torch.no_grad():
            loss_dict = model(images, targets)
        total_loss += sum(loss_dict.values()).item()
    return total_loss / len(loader)


history = {'train_loss': [], 'val_loss': [], 'lr': []}
best_val_loss = float('inf')

for epoch in range(1, EPOCHS_TOTAL + 1):
    # Unfreeze backbone after EPOCHS_FREEZE
    if epoch == EPOCHS_FREEZE + 1:
        print('\nUnfreezing backbone for fine-tuning...')
        for param in model.parameters():
            param.requires_grad = True
        optimizer = optim.SGD(model.parameters(), lr=0.001,
                              momentum=0.9, weight_decay=0.0005)
        lr_scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

    train_loss = train_one_epoch(model, optimizer, train_loader, DEVICE, epoch)
    val_loss   = evaluate_loss(model, val_loader, DEVICE)
    lr_scheduler.step()

    current_lr = optimizer.param_groups[0]['lr']
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['lr'].append(current_lr)

    print(f'Epoch {epoch:02d}/{EPOCHS_TOTAL} | '
          f'train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | lr={current_lr:.6f}')

    # Последний чекпоинт каждую эпоху (resume / копия на Drive)
    last_path = WEIGHTS_DIR / 'faster_rcnn_last.pth'
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'lr_scheduler_state_dict': lr_scheduler.state_dict(),
        'val_loss': val_loss,
        'train_loss': train_loss,
        'epochs_freeze': EPOCHS_FREEZE,
        'epochs_total': EPOCHS_TOTAL,
    }, last_path)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        ckpt_path = WEIGHTS_DIR / 'faster_rcnn_best.pth'
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'lr_scheduler_state_dict': lr_scheduler.state_dict(),
            'val_loss': val_loss,
        }, ckpt_path)
        print(f'  ✓ Best model saved (val_loss={val_loss:.4f})')

print('\nTraining complete!')

In [ ]:
# ── CELL 8: Plot training curves ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Faster R-CNN Training Curves', fontsize=13, fontweight='bold')

epochs_range = range(1, len(history['train_loss']) + 1)
axes[0].plot(epochs_range, history['train_loss'], 'b-o', markersize=4, label='Train')
axes[0].plot(epochs_range, history['val_loss'],   'r-o', markersize=4, label='Val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].semilogy(epochs_range, history['lr'], 'g-')
axes[1].set_title('Learning Rate')
axes[1].set_xlabel('Epoch')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'faster_rcnn_curves.png', dpi=150)
plt.show()

## 4. Оценка на тестовой выборке (COCO mAP)

In [ ]:
# ── CELL 9: COCO evaluation ───────────────────────────────────────────────────
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


def _per_class_ap_from_eval(coco_eval: COCOeval) -> tuple[list[float], list[float]]:
    """Per-category AP@0.5 and AP@0.5:0.95 (COCO precision tensor)."""
    precision = coco_eval.eval['precision']
    z = [0.0] * len(coco_eval.params.catIds)
    if precision is None:
        return z, z
    T, _, K, _, _ = precision.shape
    area_idx, maxdet_idx = 0, 2  # all areas, maxDets=100
    ap50: list[float] = []
    ap5095: list[float] = []
    for k in range(K):
        s0 = precision[0, :, k, area_idx, maxdet_idx]
        s0 = s0[s0 > -1]
        ap50.append(float(np.mean(s0)) if len(s0) else 0.0)
        tiers: list[float] = []
        for ti in range(T):
            s = precision[ti, :, k, area_idx, maxdet_idx]
            s = s[s > -1]
            tiers.append(float(np.mean(s)) if len(s) else 0.0)
        ap5095.append(float(np.mean(tiers)))
    return ap50, ap5095


def _align_per_class(cat_ids: list[int], vals: list[float], n: int = len(CLASS_NAMES)) -> list[float]:
    """Align values to CLASS_NAMES order (COCO category_id = 0..n-1)."""
    out = [0.0] * n
    for cid, v in zip(cat_ids, vals):
        ci = int(cid)
        if 0 <= ci < n:
            out[ci] = v
    return out


def evaluate_coco(model: torch.nn.Module, test_ds: DroneCocoDataset,
                  ann_file: Path, device: torch.device,
                  conf_thresh: float = 0.5) -> dict:
    """Run COCO evaluation on test set.

    Args:
        model: Trained model.
        test_ds: Test dataset.
        ann_file: COCO annotation JSON path.
        device: Compute device.
        conf_thresh: Confidence threshold for predictions.

    Returns:
        Dict with mAP50, mAP50_95, per-class AP, recall (COCO AR@100).
    """
    model.eval()
    coco_gt = COCO(str(ann_file))
    results = []

    test_loader = DataLoader(test_ds, batch_size=4, shuffle=False,
                             num_workers=8, collate_fn=collate_fn, pin_memory=True)

    with torch.no_grad():
        for images, targets in tqdm(test_loader, desc='COCO eval'):
            images = [img.to(device) for img in images]
            preds  = model(images)
            for pred, target in zip(preds, targets):
                img_id = target['image_id'].item()
                for box, score, label in zip(pred['boxes'], pred['scores'], pred['labels']):
                    if score < conf_thresh:
                        continue
                    x1, y1, x2, y2 = box.cpu().tolist()
                    results.append({
                        'image_id':    img_id,
                        'category_id': label.item() - 1,  # -1 to match COCO 0-indexed
                        'bbox': [x1, y1, x2 - x1, y2 - y1],
                        'score': score.item(),
                    })

    z4 = [0.0, 0.0, 0.0, 0.0]
    if not results:
        print('No predictions above threshold!')
        return {
            'mAP50': 0.0,
            'mAP50_95': 0.0,
            'per_class_mAP50': z4.copy(),
            'per_class_mAP50_95': z4.copy(),
            'recall': 0.0,
        }

    coco_dt = coco_gt.loadRes(results)
    coco_eval = COCOeval(coco_gt, coco_dt, 'bbox')
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    ap50_raw, ap5095_raw = _per_class_ap_from_eval(coco_eval)
    cat_ids = list(coco_eval.params.catIds)
    per50 = _align_per_class(cat_ids, ap50_raw)
    per5095 = _align_per_class(cat_ids, ap5095_raw)

    return {
        'mAP50_95': round(float(coco_eval.stats[0]), 4),
        'mAP50': round(float(coco_eval.stats[1]), 4),
        'per_class_mAP50': [round(x, 4) for x in per50],
        'per_class_mAP50_95': [round(x, 4) for x in per5095],
        'recall': round(float(coco_eval.stats[8]), 4),  # COCO AR@100 (не mp/mr YOLO)
    }


# Load best weights
ckpt = torch.load(WEIGHTS_DIR / 'faster_rcnn_best.pth', map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
print(f'Loaded checkpoint from epoch {ckpt["epoch"]}')

test_ds_eval = DroneCocoDataset(
    img_dir=COCO_DIR / 'images' / 'test',
    ann_file=COCO_DIR / 'annotations' / 'instances_test.json',
)

rcnn_metrics = evaluate_coco(model, test_ds_eval,
                              COCO_DIR / 'annotations' / 'instances_test.json',
                              DEVICE)
print('\nFaster R-CNN test metrics:')
for k, v in rcnn_metrics.items():
    print(f'  {k}: {v}')

In [ ]:
# ── CELL 10: Measure FPS + стабильный список тестовых кадров ───────────────────
# Сортировка по имени — как в YOLO-ноутбуках; срез для визуализации задаётся в CELL 11.
_TEST_DIR = COCO_DIR / 'images' / 'test'
TEST_IMGS_SORTED = sorted(_TEST_DIR.glob('*.jpg'), key=lambda p: p.name)

model.eval()
test_imgs = TEST_IMGS_SORTED[:50]
dummy_tensor = [transforms.ToTensor()(Image.open(p).convert('RGB')).to(DEVICE)
                for p in test_imgs[:4]]

# Warmup
with torch.no_grad():
    for _ in range(5):
        model(dummy_tensor)

start = time.perf_counter()
N = min(50, len(test_imgs))
with torch.no_grad():
    for p in test_imgs[:N]:
        img_t = [transforms.ToTensor()(Image.open(p).convert('RGB')).to(DEVICE)]
        model(img_t)
elapsed = time.perf_counter() - start
fps_rcnn = round(N / elapsed, 1) if elapsed > 0 else 0.0
print(f'Faster R-CNN FPS (single image, T4): {fps_rcnn}')

In [ ]:
# ── CELL 11: Save metrics & visualize predictions ─────────────────────────────
# Как в YOLO: срез по отсортированному списку (пример: slice(1758, 1783) или slice(0, 6))
TEST_IMG_SLICE = slice(0, 6)
VIZ_TEST_IMGS = TEST_IMGS_SORTED[TEST_IMG_SLICE]
if not VIZ_TEST_IMGS:
    raise FileNotFoundError(f'Нет jpg в {_TEST_DIR}')

rcnn_results = {
    **rcnn_metrics,
    'fps': fps_rcnn,
    'size_mb': round((WEIGHTS_DIR / 'faster_rcnn_best.pth').stat().st_size / 1e6, 1),
    'class_names': CLASS_NAMES,
}

with open(RESULTS_DIR / 'rcnn_metrics.json', 'w') as f:
    json.dump(rcnn_results, f, indent=2)
print(f'Metrics saved → {RESULTS_DIR / "rcnn_metrics.json"}')

# Визуализация: VIZ_TEST_IMGS (не «первые 6 из glob»)
CLASS_COLORS_BGR = {1: (0,0,255), 2: (255,0,0), 3: (0,255,0), 4: (0,255,255)}
CLASS_NAMES_IDX  = {1: 'DRONE', 2: 'AIRPLANE', 3: 'HELICOPTER', 4: 'BIRD'}

n_viz = len(VIZ_TEST_IMGS)
n_cols = min(3, n_viz)
n_rows = int(np.ceil(n_viz / n_cols)) if n_cols else 1
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = np.atleast_1d(axes).ravel()
fig.suptitle('Faster R-CNN Predictions on Test Set', fontsize=13, fontweight='bold')

model.eval()
sample_imgs = VIZ_TEST_IMGS
with torch.no_grad():
    for ax, img_path in zip(axes[: len(sample_imgs)], sample_imgs):
        img_pil = Image.open(img_path).convert('RGB')
        img_t   = transforms.ToTensor()(img_pil).to(DEVICE)
        preds   = model([img_t])[0]

        ax.imshow(img_pil)
        for box, score, label in zip(preds['boxes'], preds['scores'], preds['labels']):
            if score < 0.5:
                continue
            x1, y1, x2, y2 = box.cpu().tolist()
            cls_name = CLASS_NAMES_IDX.get(label.item(), f'cls{label.item()}')
            color = {'DRONE':'#e74c3c','AIRPLANE':'#3498db','HELICOPTER':'#2ecc71','BIRD':'#f1c40f'}.get(cls_name, '#fff')
            rect = patches.Rectangle((x1,y1), x2-x1, y2-y1, linewidth=2,
                                      edgecolor=color, facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1-3, f'{cls_name} {score:.2f}', fontsize=7, color=color,
                    bbox=dict(facecolor='black', alpha=0.5, pad=1))
        ax.axis('off')
        ax.set_title(img_path.name[:25], fontsize=7)
    for j in range(len(sample_imgs), len(axes)):
        axes[j].set_visible(False)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'rcnn_predictions.png', dpi=120)
plt.show()
print('Done! Next step → 04_evaluation.ipynb')